In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
assert sys.version_info[:2] == (3,12), sys.version
inputs = Path('/kaggle/input')
manifests = list(inputs.rglob('wheels/manifest.json'))
assert len(manifests)==1, 'Expected one offline dependency package'
wheels = manifests[0].parent
manifest = json.loads(manifests[0].read_text())
REQUIREMENTS = ['numpy==1.26.4', 'pandas==2.2.3', 'scipy==1.15.3', 'scikit-learn==1.8.0', 'lightgbm==4.6.0', 'koolbox==0.1.3', 'optuna==4.5.0', 'polars==1.44.2', 'pyarrow==19.0.1', 'joblib==1.5.2', 'tqdm==4.67.1']
assert manifest['requirements'] == REQUIREMENTS
for name,expected in manifest['files'].items():
    assert hashlib.sha256((wheels/name).read_bytes()).hexdigest() == expected, name
checkpoints = [p.parent for p in inputs.rglob('fold555/run_status.json')]
assert len(checkpoints)==1, 'Expected one final checkpoint'
checkpoint = checkpoints[0]
for name,expected in {'train_fold555.py': 'ca5a2abdd90afe9a23c779a110202a52dc5663c8b55d610100f8417e1df2dd1d', 'environment.json': '4f030c2dcd4d990b1e4ae69e3aa67a3a0bb6a042cf046a6e297560bbd8d89646', 'run_status.json': 'dd5bdde9d8927d837df17f46f1e5fafcbd2d95002788b622d698bf0a9fd99171', 'thresholds.pkl': '97822a769b03bbb3b3adf976c762a83f5addc0a22f3d561ce385e697da8d730a', 'body_part_configurations.json': '1e97e22a5954efb34a96643cfef38648f751abb55b6576217357e9c4eb4642e6', 'meta_categories.json': '43b08e452838c605f811917644fd8584b020af32cf18522e21c87e35d4fe3dfd'}.items():
    assert hashlib.sha256((checkpoint/name).read_bytes()).hexdigest() == expected, name
datasets = [p.parent for p in inputs.rglob('test.csv') if (p.parent/'test_tracking').is_dir() and (p.parent/'train.csv').is_file()]
assert len(datasets)==1, 'Expected one competition dataset'
data = datasets[0]
process_env = os.environ.copy()
for name in ('PYTHONPATH','PYTHONHOME'):
    process_env.pop(name,None)
process_env.update(PYTHONNOUSERSITE='1',PYTHONUNBUFFERED='1',OMP_NUM_THREADS='4',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1')
def run(command):
    with subprocess.Popen([str(x) for x in command],env=process_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
        if proc.wait():
            raise RuntimeError('Inference subprocess failed: '+str(proc.returncode))
env = Path('/tmp/mabe-offline-env')
run([sys.executable,'-I','-m','venv','--without-pip',env])
python = env/'bin/python'
run([sys.executable,'-I','-m','pip','--python',python,'install','--no-index','--find-links',wheels,
     '--no-cache-dir','--disable-pip-version-check',*REQUIREMENTS])
run([sys.executable,'-I','-m','pip','--python',python,'check'])
scripts = Path('/tmp/mabe-submission')
scripts.mkdir(exist_ok=True)
for name,source in {'infer_fold555.py': '"""Verify a completed fold555 checkpoint and predict the visible competition test set.\n\nUses the exact feature functions and category vocabulary saved with the checkpoint.\nDoes not train models, create placeholder predictions, or submit to the leaderboard.\n"""\nimport argparse\nimport gc\nimport hashlib\nimport importlib.util\nimport json\nimport shutil\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\n\nCOLUMNS = [\'video_id\', \'agent_id\', \'target_id\', \'action\', \'start_frame\', \'stop_frame\']\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef load_features(checkpoint, data):\n    environment = json.loads((checkpoint/\'environment.json\').read_text())\n    for name, expected in environment[\'packages\'].items():\n        if version(name) != expected:\n            raise ValueError(f\'Install {name}=={expected} to match this checkpoint\')\n    spec = importlib.util.spec_from_file_location(\'saved_mabe_features\', checkpoint/\'train_fold555.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    categories = json.loads((checkpoint/\'meta_categories.json\').read_text())\n    module.META_CAT_CATEGORIES.update(categories)\n    module.META_CAT_KNOWN_SET.update({k: set(v) for k, v in categories.items()})\n    module.META_CAT_ENCODERS.update({k: {v:i for i,v in enumerate(values)} for k,values in categories.items()})\n    module.META_CAT_UNK.update({k: len(v) for k, v in categories.items()})\n    module.train = pd.read_csv(data/\'train.csv\')\n    module.test = pd.read_csv(data/\'test.csv\')\n    for frame in (module.train, module.test):\n        frame[\'n_mice\'] = 4-frame[[f\'mouse{i}_strain\' for i in range(1,5)]].isna().sum(axis=1)\n    configurations = json.loads((checkpoint/\'body_part_configurations.json\').read_text())\n    if list(np.unique(module.train.body_parts_tracked)) != configurations:\n        raise ValueError(\'Training metadata does not match saved section numbering\')\n    module.arena_data = pd.concat([f[[\'video_id\',\'arena_width_cm\',\'arena_height_cm\',\'arena_shape\']]\n                                  for f in (module.train,module.test)]).drop_duplicates(\'video_id\').set_index(\'video_id\')\n    return module, configurations\n\n\ndef check_probabilities(values):\n    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n        raise ValueError(\'Prediction contains invalid probabilities\')\n\n\ndef predict_bundle(bundle, features, threads=2):\n    if not bundle.is_fitted or len(bundle.estimators) != 5:\n        raise ValueError(\'Expected five fitted models\')\n    total = np.zeros(len(features), dtype=np.float64)\n    # LightGBM replaces spaces in stored feature names with underscores.\n    # Keep the checkpoint\'s original column order while comparing that encoding.\n    names = [str(c).replace(\' \', \'_\') for c in features.columns]\n    if len(names) != len(set(names)):\n        raise ValueError(\'Feature names collide after LightGBM normalization\')\n    for model in bundle.estimators:\n        if model.n_features_in_ != features.shape[1] or list(model.feature_name_) != names:\n            raise ValueError(\'Model feature names or ordering differ\')\n        if list(model.classes_) != [0, 1]:\n            raise ValueError(\'Unexpected class ordering\')\n        model.set_params(n_jobs=threads)\n        predicted = model.predict_proba(features)[:,1]\n        check_probabilities(predicted)\n        total += predicted\n    return total/len(bundle.estimators)\n\n\ndef intervals(probabilities, meta, thresholds):\n    """Original argmax/threshold rule, preserving the final interval and frame gaps."""\n    if not len(meta) or not len(probabilities.columns):\n        return pd.DataFrame(columns=COLUMNS)\n    if len(probabilities) != len(meta):\n        raise ValueError(\'Prediction and frame counts differ\')\n    identities = meta[[\'video_id\',\'agent_id\',\'target_id\']].drop_duplicates()\n    if len(identities) != 1:\n        raise ValueError(\'Convert one video/agent/target at a time\')\n    values = probabilities.to_numpy()\n    check_probabilities(values)\n    frames = meta.video_frame.to_numpy(dtype=np.int64)\n    if (np.diff(frames) <= 0).any():\n        raise ValueError(\'Frame indices must be strictly increasing\')\n    winner = np.argmax(values, axis=1)\n    cutoffs = np.array([thresholds[a] for a in probabilities.columns])\n    labels = np.where(values[np.arange(len(values)),winner] >= cutoffs[winner],winner,-1)\n    starts = np.flatnonzero(np.r_[True, (labels[1:] != labels[:-1]) | (np.diff(frames) != 1)])\n    ends = np.r_[starts[1:],len(frames)]\n    identity = identities.iloc[0]\n    rows = [(int(identity.video_id),str(identity.agent_id),str(identity.target_id),\n             str(probabilities.columns[labels[start]]),int(frames[start]),int(frames[end-1]+1))\n            for start,end in zip(starts,ends) if labels[start] >= 0]\n    return pd.DataFrame(rows, columns=COLUMNS)\n\n\ndef audit(checkpoint, thresholds, output, threads, audit_oof):\n    rows = []\n    for marker in sorted(checkpoint.glob(\'*/*/task_complete.json\')):\n        task = json.loads(marker.read_text())\n        folder = marker.parent\n        for name, expected in task[\'sha256\'].items():\n            if digest(folder/name) != expected:\n                raise ValueError(f\'Corrupted checkpoint: {folder.name}/{name}\')\n        score = task[\'score\']\n        section,kind,action = str(score[\'section\']),score[\'kind\'],score[\'action\']\n        cutoff = float(joblib.load(folder/\'threshold.pkl\'))\n        if cutoff != thresholds[kind][section][action] or not 0 <= cutoff <= 1:\n            raise ValueError(\'Threshold files disagree\')\n        columns = json.loads((folder/\'feature_columns.json\').read_text())\n        model_files = list(folder.glob(\'*_trainer_*.pkl\'))\n        if len(model_files) != 1 or len(columns) != len(set(columns)):\n            raise ValueError(\'Ambiguous model or duplicated columns\')\n        bundle = joblib.load(model_files[0])\n        # Synthetic inputs check serialization/predictability only, not model accuracy.\n        synthetic = pd.DataFrame(np.vstack([np.zeros(len(columns)),np.full(len(columns),np.nan)]),columns=columns)\n        prediction = predict_bundle(bundle, synthetic, threads)\n        row = dict(section=section,kind=kind,action=action,folds=len(bundle.estimators),features=len(columns),\n                   threshold=cutoff,smoke_test=\'passed\',saved_binary_f1=score[\'binary F1 score\'])\n        if audit_oof:\n            tp = fp = fn = count = 0\n            folds_seen = set()\n            for batch in pq.ParquetFile(folder/\'oof_predictions.parquet\').iter_batches(\n                    batch_size=65536,columns=[\'label\',\'prediction\',\'fold\']):\n                frame = batch.to_pandas()\n                y,p = frame.label.to_numpy(),frame.prediction.to_numpy()\n                check_probabilities(p)\n                if not np.isin(y,[0,1]).all() or not np.isin(frame.fold,[0,1,2,3,4]).all():\n                    raise ValueError(\'Invalid out-of-fold labels or fold indices\')\n                predicted = p >= cutoff\n                tp += int(((y == 1)&predicted).sum())\n                fp += int(((y == 0)&predicted).sum())\n                fn += int(((y == 1)&~predicted).sum())\n                count += len(frame)\n                folds_seen.update(frame.fold.unique().tolist())\n            f1 = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0\n            if folds_seen != set(range(5)) or not np.isclose(f1,score[\'binary F1 score\'],atol=1e-12):\n                raise ValueError(f\'Saved validation score mismatch: {section}/{action}\')\n            row.update(oof_rows=count,recomputed_binary_f1=f1)\n        rows.append(row)\n        del bundle,synthetic,prediction\n        gc.collect()\n        if len(rows)%10 == 0:\n            print(f\'AUDIT {len(rows)} tasks verified\',flush=True)\n    if len(rows) != json.loads((checkpoint/\'run_status.json\').read_text())[\'trained_tasks\']:\n        raise ValueError(\'Task inventory is incomplete\')\n    pd.DataFrame(rows).to_csv(output/\'model-audit.csv\',index=False)\n    return rows\n\n\ndef validate_submission(submission, dataset, data):\n    if list(submission.columns) != COLUMNS or submission.empty:\n        raise ValueError(\'No genuine model predictions were generated\')\n    if submission.isna().any().any() or submission.duplicated().any():\n        raise ValueError(\'Missing values or duplicate prediction rows\')\n    for row in dataset.itertuples():\n        sample = submission[submission.video_id == row.video_id]\n        if sample.empty:\n            continue\n        allowed = {tuple(x.replace("\'",\'\').split(\',\')) for x in json.loads(row.behaviors_labeled)}\n        tracking = pq.read_table(data/\'test_tracking\'/row.lab_id/f\'{row.video_id}.parquet\',columns=[\'video_frame\'])\n        frames = tracking[\'video_frame\'].to_numpy()\n        if not all((r.agent_id,r.target_id,r.action) in allowed for r in sample.itertuples()):\n            raise ValueError(\'Prediction uses an unlabelled behavior or mouse pair\')\n        if ((sample.start_frame < frames.min()) | (sample.stop_frame > frames.max()+1)\n                | (sample.start_frame >= sample.stop_frame)).any():\n            raise ValueError(\'Prediction interval is outside the video\')\n    if not set(submission.video_id) <= set(dataset.video_id):\n        raise ValueError(\'Unknown video in submission\')\n    for _, group in submission.groupby([\'video_id\',\'agent_id\',\'target_id\']):\n        group = group.sort_values(\'start_frame\')\n        if (group.start_frame.to_numpy()[1:] < group.stop_frame.to_numpy()[:-1]).any():\n            raise ValueError(\'Overlapping prediction intervals\')\n\n\ndef run(checkpoint, data, output, threads=2, audit_oof=False, export_inputs=False):\n    checkpoint,data,output = map(Path,(checkpoint,data,output))\n    output.mkdir(parents=True,exist_ok=True)\n    if json.loads((checkpoint/\'run_status.json\').read_text())[\'status\'] != \'complete\':\n        raise ValueError(\'Checkpoint training is not complete\')\n    module,configurations = load_features(checkpoint,data)\n    thresholds = joblib.load(checkpoint/\'thresholds.pkl\')\n    audits = audit(checkpoint,thresholds,output,threads,audit_oof)\n    predictions,coverage,record_predictions = [],[],[]\n    legacy_rows = 0\n    for row in module.test.itertuples():\n        if row.body_parts_tracked not in configurations:\n            raise ValueError(f\'Unseen body part configuration for video {row.video_id}\')\n        section = configurations.index(row.body_parts_tracked)\n        subset = module.test[module.test.video_id == row.video_id]\n        parts = json.loads(row.body_parts_tracked)\n        if len(parts)>5:\n            parts = [p for p in parts if p not in module.drop_body_parts]\n        expected = {tuple(x.replace("\'",\'\').split(\',\')) for x in json.loads(row.behaviors_labeled)}\n        observed = set()\n        for kind,tracking,meta,actions in module.generate_mouse_data(subset,\'test\',str(data/\'test_tracking\')):\n            agent,target = str(meta.agent_id.iloc[0]),str(meta.target_id.iloc[0])\n            relevant = [a for a in actions if a in thresholds.get(kind,{}).get(str(section),{})]\n            if not relevant:\n                continue\n            features,_,_ = module.make_features(kind,tracking,meta,parts,section)\n            probabilities = pd.DataFrame(index=np.arange(len(meta)))\n            for action in relevant:\n                folder = checkpoint/str(section)/str(action)\n                columns = json.loads((folder/\'feature_columns.json\').read_text())\n                model = joblib.load(next(folder.glob(\'*_trainer_*.pkl\')))\n                aligned = features.reindex(columns=columns)\n                probabilities[action] = predict_bundle(model,aligned,threads)\n                observed.add((agent,target,str(action)))\n                record_predictions.append(dict(video_id=int(row.video_id),section=section,kind=kind,\n                    agent_id=agent,target_id=target,action=str(action),frames=len(meta),\n                    expected_features=len(columns),missing_features=len(set(columns)-set(features.columns)),\n                    min_probability=float(probabilities[action].min()),max_probability=float(probabilities[action].max())))\n                del model,aligned\n                gc.collect()\n            limits = thresholds[kind][str(section)]\n            original = module.predict_multiclass(probabilities,meta,limits)\n            legacy_rows += len(original)\n            predictions.append(intervals(probabilities,meta,limits))\n            # Retain real frame probabilities so both exports can be inspected/reproduced.\n            trace = meta[[\'video_id\',\'agent_id\',\'target_id\',\'video_frame\']].reset_index(drop=True)\n            trace = pd.concat([trace,probabilities.add_prefix(\'probability_\')],axis=1)\n            trace.to_parquet(output/f\'probabilities-{row.video_id}-{agent}-{target}.parquet\',index=False)\n            del features,tracking,meta,probabilities,trace\n            gc.collect()\n        missing = sorted(expected-observed)\n        coverage.append(dict(video_id=int(row.video_id),expected_behaviors=len(expected),\n                             predicted_behaviors=len(observed),unavailable_models=missing))\n        print(f\'PREDICT video={row.video_id}, behavior coverage={len(observed)}/{len(expected)}\',flush=True)\n    result = pd.concat(predictions,ignore_index=True) if predictions else pd.DataFrame(columns=COLUMNS)\n    validate_submission(result,module.test,data)\n    result = result.sort_values([\'video_id\',\'agent_id\',\'target_id\',\'start_frame\']).reset_index(drop=True)\n    result.index.name = \'row_id\'\n    result.to_csv(output/\'submission.csv\')\n    reloaded = pd.read_csv(output/\'submission.csv\')\n    assert list(reloaded.columns) == [\'row_id\']+COLUMNS\n    assert reloaded.row_id.tolist() == list(range(len(reloaded)))\n    report = dict(status=\'passed\',checkpoint_tasks=len(audits),checkpoint_folds=sum(r[\'folds\'] for r in audits),\n                  visible_test_videos=len(module.test),predicted_videos=int(result.video_id.nunique()),\n                  prediction_rows=len(result),coverage=coverage,prediction_records=record_predictions,\n                  original_interval_rows=legacy_rows,interval_export=\'argmax + saved thresholds; final interval retained; gaps split\',\n                  oof_scores_recomputed=audit_oof,submission_sha256=digest(output/\'submission.csv\'),\n                  scope=\'Visible competition test data only; no hidden-test or leaderboard evaluation\')\n    (output/\'inference-verification.json\').write_text(json.dumps(report,indent=2))\n    if export_inputs:\n        target = output/\'verification_inputs\'\n        target.mkdir(exist_ok=True)\n        for name in (\'train.csv\',\'test.csv\',\'sample_submission.csv\'):\n            if (data/name).exists():\n                shutil.copy2(data/name,target/name)\n        for row in module.test.itertuples():\n            path = Path(\'test_tracking\')/row.lab_id/f\'{row.video_id}.parquet\'\n            (target/path).parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(data/path,target/path)\n    print(\'INFERENCE VERIFIED:\',json.dumps({k:report[k] for k in (\'checkpoint_tasks\',\'checkpoint_folds\',\'visible_test_videos\',\'prediction_rows\')}),flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--checkpoint\',type=Path,required=True)\n    parser.add_argument(\'--data\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--threads\',type=int,default=2)\n    parser.add_argument(\'--audit-oof\',action=\'store_true\')\n    parser.add_argument(\'--export-inputs\',action=\'store_true\')\n    args = parser.parse_args()\n    run(args.checkpoint,args.data,args.output,args.threads,args.audit_oof,args.export_inputs)\n', 'submit_fold555.py': '"""Offline competition inference using the verified final checkpoint."""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\nfrom types import SimpleNamespace\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom infer_fold555 import COLUMNS, digest, intervals, load_features, predict_bundle, validate_submission\n\n\ndef load_bundle(folder, kind, cutoff):\n    marker = json.loads((folder/\'task_complete.json\').read_text())\n    if marker[\'score\'][\'kind\'] != kind:\n        raise ValueError(f\'Wrong model kind: {folder}\')\n    files = list(folder.glob(\'*_trainer_*.pkl\'))\n    if len(files) != 1:\n        raise ValueError(f\'Ambiguous model: {folder}\')\n    for name in (files[0].name,\'feature_columns.json\',\'threshold.pkl\'):\n        if digest(folder/name) != marker[\'sha256\'][name]:\n            raise ValueError(f\'Corrupted checkpoint: {folder}/{name}\')\n    if float(joblib.load(folder/\'threshold.pkl\')) != cutoff:\n        raise ValueError(\'Saved threshold files disagree\')\n    columns = json.loads((folder/\'feature_columns.json\').read_text())\n    original = joblib.load(files[0])\n    # Retain only inference models, releasing OOF arrays and training group IDs.\n    compact = SimpleNamespace(is_fitted=original.is_fitted,estimators=original.estimators)\n    del original\n    gc.collect()\n    return compact,columns\n\n\ndef run(checkpoint, data, output, threads=4):\n    checkpoint,data,output = map(Path,(checkpoint,data,output))\n    status = json.loads((checkpoint/\'run_status.json\').read_text())\n    if status[\'status\'] != \'complete\' or status[\'trained_tasks\'] != 84:\n        raise ValueError(\'Expected the completed 84-task checkpoint\')\n    module,configurations = load_features(checkpoint,data)\n    thresholds = joblib.load(checkpoint/\'thresholds.pkl\')\n    output.mkdir(parents=True,exist_ok=True)\n    csv = output/\'submission.csv\'\n    pd.DataFrame(columns=[\'row_id\']+COLUMNS).to_csv(csv,index=False)\n    row_count = predicted_videos = records = 0\n    coverage = []\n    cache = {}\n    cached_section = None\n    start = perf_counter()\n    # Group by keypoint configuration to reuse small, inference-only model bundles.\n    for row in module.test.sort_values([\'body_parts_tracked\',\'video_id\']).itertuples():\n        if row.body_parts_tracked not in configurations:\n            raise ValueError(f\'Unseen body part configuration: video {row.video_id}\')\n        section = configurations.index(row.body_parts_tracked)\n        if section != cached_section:\n            cache.clear()\n            gc.collect()\n            cached_section = section\n        subset = module.test[module.test.video_id == row.video_id]\n        parts = json.loads(row.body_parts_tracked)\n        if len(parts)>5:\n            parts = [p for p in parts if p not in module.drop_body_parts]\n        expected = {tuple(x.replace("\'",\'\').split(\',\')) for x in json.loads(row.behaviors_labeled)}\n        observed = set()\n        predictions = []\n        for kind,tracking,meta,actions in module.generate_mouse_data(subset,\'test\',str(data/\'test_tracking\')):\n            limits = thresholds.get(kind,{}).get(str(section),{})\n            relevant = [a for a in actions if a in limits]\n            if not relevant:\n                continue\n            features,_,_ = module.make_features(kind,tracking,meta,parts,section)\n            probabilities = pd.DataFrame(index=np.arange(len(meta)))\n            for action in relevant:\n                key = (kind,str(action))\n                if key not in cache:\n                    cache[key] = load_bundle(checkpoint/str(section)/str(action),kind,limits[action])\n                bundle,columns = cache[key]\n                aligned = features.reindex(columns=columns)\n                probabilities[action] = predict_bundle(bundle,aligned,threads)\n                observed.add((str(meta.agent_id.iloc[0]),str(meta.target_id.iloc[0]),str(action)))\n                del aligned\n            predictions.append(intervals(probabilities,meta,limits))\n            records += 1\n            del features,tracking,meta,probabilities\n            gc.collect()\n        result = pd.concat(predictions,ignore_index=True) if predictions else pd.DataFrame(columns=COLUMNS)\n        if not result.empty:\n            validate_submission(result,subset,data)\n            result = result.sort_values([\'video_id\',\'agent_id\',\'target_id\',\'start_frame\']).reset_index(drop=True)\n            result.insert(0,\'row_id\',np.arange(row_count,row_count+len(result)))\n            result.to_csv(csv,mode=\'a\',header=False,index=False)\n            row_count += len(result)\n            predicted_videos += 1\n        coverage.append(dict(video_id=int(row.video_id),section=section,\n            expected_behaviors=len(expected),predicted_behaviors=len(observed),\n            unavailable_models=sorted(expected-observed),prediction_rows=len(result)))\n        print(f\'PREDICT video={row.video_id}, section={section}, behaviors={len(observed)}/{len(expected)}, \'\n              f\'intervals={len(result)}, completed={len(coverage)}/{len(module.test)}, elapsed={perf_counter()-start:.1f}s\',flush=True)\n        del predictions,result\n        gc.collect()\n    if not row_count:\n        raise ValueError(\'No genuine predictions were generated\')\n    report = dict(status=\'passed\',checkpoint_tasks=84,checkpoint_folds=420,\n        input_videos=len(module.test),predicted_videos=predicted_videos,prediction_records=records,\n        prediction_rows=row_count,coverage=coverage,submission_sha256=digest(csv),\n        elapsed_seconds=perf_counter()-start,scope=\'Predictions only; competition scoring is performed by Kaggle\')\n    (output/\'submission-report.json\').write_text(json.dumps(report,indent=2))\n    print(\'SUBMISSION READY:\',json.dumps({k:v for k,v in report.items() if k!=\'coverage\'}),flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--checkpoint\',type=Path,required=True)\n    parser.add_argument(\'--data\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--threads\',type=int,default=4)\n    args = parser.parse_args()\n    run(args.checkpoint,args.data,args.output,args.threads)\n'}.items():
    (scripts/name).write_text(source)
arguments = [str(scripts/'submit_fold555.py'),'--checkpoint',str(checkpoint),'--data',str(data),
             '--output','/kaggle/working','--threads','4']
launcher = 'import sys,runpy; sys.path.insert(0,'+repr(str(scripts))+'); sys.argv='+repr(arguments)+'; runpy.run_path(sys.argv[0],run_name="__main__")'
run([python,'-I','-u','-c',launcher])
assert Path('/kaggle/working/submission.csv').is_file()
